# NB138: The R₀ Mechanism

**Goal**: Understand WHY the base-level exponents x_q(R₀) ≈ 4/7 and x_l(R₀) ≈ 27/11 arise, and HOW the cascade amplifies them to exact rationals at R₃.

**Background** (NB137 S26–S28): The window-0 mass exponent factors as x(R₃) = x(R₀) × cross-level:
- Lepton: (27/11)(11/9) = 3 = p₂ [identity #277]
- Quark: (4/7)(25/9) = 100/63

All six quantities are T-independent structural invariants.

In [2]:
# ── S0: Setup ──

import sys, os, time, warnings
import numpy as np
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               DLOG, PHYSICAL_CROSSINGS, CP_PAIRS,
                               SM_TARGETS, ACTIVE_PRIMES)
from solenoid_system import SolenoidSystem

p1, p2, p3, p4 = SA.primes
P4 = SA.P
PHI_P4 = SA.PHI

def target_value(name):
    return SM_TARGETS[name][0]

M_MU_E = target_value('m_mu/m_e')
M_S_D  = target_value('m_s/m_d')

sys0 = SolenoidSystem()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

print(f'P4={P4}, phi={PHI_P4}, primes={SA.primes}')
print(f'kappa = epsilon = 1/sqrt(210) = {KAPPA:.6f}')
print(f'omega = 2*pi = {OMEGA:.6f}')
print(f'Window-0: {len(cis)} coprime crossings in [1, {P4})')
print(f'M_s/M_d = {M_S_D}, M_mu/M_e = {M_MU_E}')

P4=210, phi=48, primes=[2, 3, 5, 7]
kappa = epsilon = 1/sqrt(210) = 0.069007
omega = 2*pi = 6.283185
Window-0: 48 coprime crossings in [1, 210)
M_s/M_d = 20.0, M_mu/M_e = 206.768

## S1: The R₀ Analytic Solution

R₀ obeys an autonomous linear ODE: $dR_0/dt + \kappa R_0 = \varepsilon \sin(\omega t)$

**Critical simplification**: $\omega = 2\pi$ and crossings are at integer times, so $\sin(2\pi \cdot \text{ci}) = 0$, $\cos(2\pi \cdot \text{ci}) = 1$. The oscillatory steady state vanishes. At coprime crossings:

$$R_0(\text{ci}; j_1) = (2\pi j_1 + D)\, e^{-\kappa \cdot \text{ci}} - D \qquad\text{where } D = \varepsilon\omega/(\omega^2+\kappa^2)$$

In [4]:
# ── S1: R₀ analytic formula and verification ──

print('S1: R0 ANALYTIC SOLUTION AT INTEGER CROSSINGS')
print('='*70)

D = EPSILON * OMEGA / (OMEGA**2 + KAPPA**2)
C1 = 2 * np.pi + D

print(f'  D = eps*omega/(omega^2+kappa^2) = {D:.8e}')
print(f'  C1 = 2*pi + D                  = {C1:.8f}')
print(f'  1/kappa (e-folding time)        = {1/KAPPA:.2f}')

def R0_at_crossing(ci, j1):
    alpha = np.exp(-KAPPA * ci)
    return (2 * np.pi * j1 + D) * alpha - D

# Verify against full solution (including sin/cos terms)
A_coeff = EPSILON * KAPPA / (OMEGA**2 + KAPPA**2)
B_coeff = -EPSILON * OMEGA / (OMEGA**2 + KAPPA**2)
print('\nVerification: simplified vs full solution at integer crossings')
for ci_test in [1, 11, 31, 61, 191]:
    simplified = R0_at_crossing(float(ci_test), 0)
    full = (-B_coeff)*np.exp(-KAPPA*ci_test) + A_coeff*np.sin(OMEGA*ci_test) + B_coeff*np.cos(OMEGA*ci_test)
    print(f'  ci={ci_test:3d}: simplified={simplified:.12f}, full={full:.12f}, diff={simplified-full:.2e}')

# Verify against numerical ODE integration
print('\nVerification: analytic vs scipy ODE')
t_cross = cis.astype(float)
T_integ = float(P4 + 1)
res_j0 = sys0.integrate_branch((0,0,0,0), t_cross, T_integ)
res_j1 = sys0.integrate_branch((1,0,0,0), t_cross, T_integ)

max_dev_0 = max(abs(R0_at_crossing(float(cis[i]), 0) - res_j0[i, 0]) for i in range(len(cis)))
max_dev_1 = max(abs(R0_at_crossing(float(cis[i]), 1) - res_j1[i, 0]) for i in range(len(cis)))
print(f'  Max |analytic - numerical|: j1=0: {max_dev_0:.2e}, j1=1: {max_dev_1:.2e}')
print(f'  -> Analytic formula is EXACT (to ODE solver precision)')

print(f'\nR0 at physical crossings:')
print(f'  {"ci":>4s} {"R0(j1=0)":>12s} {"R0(j1=1)":>12s} {"transient":>12s} {"trans/D":>8s}')
for ci_val in [11, 31, 61, 191]:
    r0 = R0_at_crossing(float(ci_val), 0)
    r1 = R0_at_crossing(float(ci_val), 1)
    trans = C1 * np.exp(-KAPPA * ci_val)
    print(f'  {ci_val:4d} {r0:12.6f} {r1:12.6f} {trans:12.6f} {trans/D:8.1f}')

S1: R0 ANALYTIC SOLUTION AT INTEGER CROSSINGS
  D = eps*omega/(omega^2+kappa^2) = 1.09814099e-02
  C1 = 2*pi + D                  = 6.29416672
  1/kappa (e-folding time)        = 14.49

Verification: simplified vs full solution at integer crossings
  ci=  1: simplified=-0.000732234249, full=-0.000732234249, diff=0.00e+00
  ci= 11: simplified=-0.005841005678, full=-0.005841005678, diff=8.67e-19
  ci= 31: simplified=-0.009688363997, full=-0.009688363997, diff=1.73e-18
  ci= 61: simplified=-0.010818277980, full=-0.010818277980, diff=1.73e-18
  ci=191: simplified=-0.010981389172, full=-0.010981389172, diff=1.73e-18

Verification: analytic vs scipy ODE
  Max |analytic - numerical|: j1=0: 3.71e-14, j1=1: 3.61e-13
  -> Analytic formula is EXACT (to ODE solver precision)

R0 at physical crossings:
    ci     R0(j1=0)     R0(j1=1)    transient  trans/D
    11    -0.005841     2.935322     2.946303    268.3
    31    -0.009688     0.730148     0.741129     67.5
    61    -0.010818     0.082520  

## S2: CRT Sector Structure — One Crossing Per Sector

The 48 coprime integers mod 210 biject onto Z*₂₁₀. The CP pair ratio at R₀ compares exactly TWO crossing times: g and g' = g⁻¹ mod 210. The gap between them is determined by CRT: both share residues mod 2,3,5 and differ only mod 7.

In [6]:
# ── S2: CRT sector map ──

print('S2: CRT SECTOR STRUCTURE')
print('='*70)

sector_ci = {}
for idx in range(len(cis)):
    sector_ci[(a3[idx], a5[idx], a7[idx])] = cis[idx]

print(f'Total sectors: {len(sector_ci)} (= phi(210) = 48)')
print(f'All have exactly 1 crossing: {len(sector_ci) == PHI_P4}')

print('\na5=0 sectors (used for CP pair ratios):')
print(f'  {"(a3,a7)":>8s} {"ci":>5s} {"mod3":>5s} {"mod5":>5s} {"mod7":>5s} {"role":>15s}')
for a3v in [0, 1]:
    for a7v in range(6):
        key = (a3v, 0, a7v)
        if key in sector_ci:
            cv = sector_ci[key]
            role = ''
            if (a3v, a7v) == (1, 4): role = 'QUARK g1'
            elif (a3v, a7v) == (1, 2): role = 'QUARK g2'
            elif (a3v, a7v) == (0, 1): role = 'LEPTON g1'
            elif (a3v, a7v) == (0, 5): role = 'LEPTON g2'
            print(f'  ({a3v},{a7v})   {cv:5d} {cv%3:5d} {cv%5:5d} {cv%7:5d} {role:>15s}')

print('\nCP pair verification (g * g\' = 1 mod 210):')
for name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    g = sector_ci[(ch_a3, 0, a7g1)]
    gp = sector_ci[(ch_a3, 0, a7g2)]
    print(f'  {name:>7s}: g={g:3d}, g\'={gp:3d}, g*g\' mod 210 = {(g*gp)%210}, gap = {gp-g}')

print('\nGap structure:')
print('  LEPTON gap = 61-31 = 30 = P3 (third primorial)')
print('  QUARK  gap = 191-11 = 180 = 6*P3')
print('  Both pairs differ only in a7 (mod 7 residue).')
print('  Gap = 30*k where k solves 30*k = (g2-g1) mod 7')
for name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    g = sector_ci[(ch_a3, 0, a7g1)]
    gp = sector_ci[(ch_a3, 0, a7g2)]
    k = (gp - g) // 30
    print(f'  {name}: 30*{k} mod 7 = {(30*k)%7} = ({gp%7}-{g%7}) mod 7 = {(gp-g)%7}')

S2: CRT SECTOR STRUCTURE
Total sectors: 48 (= phi(210) = 48)
All have exactly 1 crossing: True

a5=0 sectors (used for CP pair ratios):
   (a3,a7)    ci  mod3  mod5  mod7            role
  (0,0)       1     1     1     1                
  (0,1)      31     1     1     3       LEPTON g1
  (0,2)     121     1     1     2                
  (0,3)     181     1     1     6                
  (0,4)     151     1     1     4                
  (0,5)      61     1     1     5       LEPTON g2
  (1,0)      71     2     1     1                
  (1,1)     101     2     1     3                
  (1,2)     191     2     1     2        QUARK g2
  (1,3)      41     2     1     6                
  (1,4)      11     2     1     4        QUARK g1
  (1,5)     131     2     1     5                

CP pair verification (g * g' = 1 mod 210):
    QUARK: g= 11, g'=191, g*g' mod 210 = 1, gap = 180
   LEPTON: g= 31, g'= 61, g*g' mod 210 = 1, gap = 30

Gap structure:
  LEPTON gap = 61-31 = 30 = P3 (third primoria

## S3: The R₀ CP Ratio — Exact Analytic Formula

Since each a₅=0 sector has one crossing, RMS(ci) = √(½(R₀(ci;0)² + R₀(ci;1)²)). The CP ratio is C₀(R₀) = RMS(g)/RMS(g'). We compute this analytically and examine WHY the quark and lepton regimes differ so drastically.

In [8]:
# ── S3: R₀ CP ratio — exact analytic ──

print('S3: R0 CP RATIO — EXACT ANALYTIC')
print('='*70)

def r0sq_avg(ci):
    """½(R₀(ci;0)² + R₀(ci;1)²) — the branch-averaged R₀²."""
    alpha = np.exp(-KAPPA * float(ci))
    r0 = D * (alpha - 1)
    r1 = C1 * alpha - D
    return 0.5 * (r0**2 + r1**2)

for name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    g = sector_ci[(ch_a3, 0, a7g1)]
    gp = sector_ci[(ch_a3, 0, a7g2)]
    c0 = np.sqrt(r0sq_avg(g) / r0sq_avg(gp))
    mass_r = M_S_D if name == 'QUARK' else M_MU_E
    x = np.log(mass_r) / np.log(c0)
    target = 4/7 if name == 'QUARK' else Fraction(27, 11)
    dev = (x / float(target) - 1) * 1e6

    print(f'\n{name} (g={g}, g\'={gp}, gap={gp-g}):')
    print(f'  RMS(g)  = {np.sqrt(r0sq_avg(g)):.8f}')
    print(f'  RMS(g\') = {np.sqrt(r0sq_avg(gp)):.8f}')
    print(f'  C0(R0)  = {c0:.8f}')
    print(f'  x(R0)   = {x:.8f}  (target {float(target):.8f}, dev = {dev:+.1f} ppm)')

    # Decompose: what sets the ratio?
    alpha_g = np.exp(-KAPPA * float(g))
    alpha_gp = np.exp(-KAPPA * float(gp))
    print(f'  alpha_g  = exp(-{g}/sqrt(210)) = {alpha_g:.6f}')
    print(f'  alpha_g\' = exp(-{gp}/sqrt(210)) = {alpha_gp:.2e}')
    if alpha_gp < 0.01:
        # g' is in SS regime: RMS(g') ≈ D
        print(f'  g\' is in SS regime: RMS(g\') ~ D = {D:.6e}')
        print(f'  C0 ~ RMS(g)/D = {np.sqrt(r0sq_avg(g))/D:.4f} (vs exact {c0:.4f})')
    else:
        # Both in transient regime: C0 ~ exp(kappa*(g'-g))
        pure_exp = np.exp(KAPPA * (gp - g))
        print(f'  Pure exponential approx: exp({gp-g}/sqrt(210)) = {pure_exp:.4f}')
        print(f'  Exact C0 = {c0:.4f}, correction = {c0/pure_exp:.4f}')

print('\n\nKEY INSIGHT:')
print('  QUARK: ci=11 (transient) vs ci=191 (SS)')
print('    -> ratio = transient_amplitude / SS_offset')
print('    -> VERY large C0 (~189), sensitive to transient decay at ci=11')
print('  LEPTON: ci=31 (transient) vs ci=61 (transient)')
print('    -> ratio ~ exp(kappa*30) = exp(30/sqrt(210))')
print('    -> Both in same regime, cleaner exponential behavior')

S3: R0 CP RATIO — EXACT ANALYTIC

QUARK (g=11, g'=191, gap=180):
  RMS(g)  = 2.07558993
  RMS(g') = 0.01097546
  C0(R0)  = 189.11186765
  x(R0)   = 0.57144958  (target 0.57142857, dev = +36.8 ppm)
  alpha_g  = exp(-11/sqrt(210)) = 0.468101
  alpha_g' = exp(-191/sqrt(210)) = 1.89e-06
  g' is in SS regime: RMS(g') ~ D = 1.098141e-02
  C0 ~ RMS(g)/D = 189.0094 (vs exact 189.1119)

LEPTON (g=31, g'=61, gap=30):
  RMS(g)  = 0.51633809
  RMS(g') = 0.05884989
  C0(R0)  = 8.77381613
  x(R0)   = 2.45495281  (target 2.45454545, dev = +166.0 ppm)
  alpha_g  = exp(-31/sqrt(210)) = 0.117749
  alpha_g' = exp(-61/sqrt(210)) = 1.49e-02
  Pure exponential approx: exp(30/sqrt(210)) = 7.9264
  Exact C0 = 8.7738, correction = 1.1069


KEY INSIGHT:
  QUARK: ci=11 (transient) vs ci=191 (SS)
    -> ratio = transient_amplitude / SS_offset
    -> VERY large C0 (~189), sensitive to transient decay at ci=11
  LEPTON: ci=31 (transient) vs ci=61 (transient)
    -> ratio ~ exp(kappa*30) = exp(30/sqrt(210))
    -> B

## S4: Per-Level Cascade Progression

R₀ sets the initial CP asymmetry via the exponential transient mechanism. Each subsequent level is driven by levels below through the coupled cascade ODE. We integrate all 210 branches and track how x(Rk) evolves through k=0,1,2,3 to understand the cross-level amplification.

In [10]:
# ── S4: Per-level cascade progression (all 210 branches) ──

from solenoid_jax import warmup as jax_warmup, detect_device

print('S4: PER-LEVEL CASCADE PROGRESSION')
print('='*70)

print(f'JAX device: {detect_device()}')
t0 = time.time()
jax_warmup()
print(f'JAX warmup: {time.time()-t0:.1f}s')

all_branches = sys0.all_branches()
t_cross = cis.astype(float)
T_integ = float(P4 + 1)

t0 = time.time()
res = sys0.integrate_all_branches(all_branches, t_cross, T_integ, backend='jax')
print(f'Integrated {len(all_branches)} branches in {time.time()-t0:.1f}s')

w0_cp, w0_rms = sys0.window0_cp_ratios(
    res, cis, a3, a5, a7, P=P4, n_levels=4, cp_pairs=CP_PAIRS
)

print(f'\nPer-level exponents:')
print(f'  {"Channel":>7s}  {"k":>2s}  {"C0(Rk)":>10s}  {"x(Rk)":>10s}  {"x(Rk)/x(R0)":>12s}')
for ch_name in ['QUARK', 'LEPTON']:
    mass_r = M_S_D if ch_name == 'QUARK' else M_MU_E
    xs = []
    for k in range(4):
        c0 = w0_cp[ch_name][k]
        x = np.log(mass_r) / np.log(c0)
        xs.append(x)
        ratio_str = f'{x/xs[0]:.8f}' if k > 0 else '1.00000000'
        print(f'  {ch_name:>7s}  R{k}  {c0:10.4f}  {x:10.6f}  {ratio_str:>12s}')

    target_cl = 25/9 if ch_name == 'QUARK' else 11/9
    actual_cl = xs[3] / xs[0]
    print(f'  {ch_name:>7s}  cross-level = {actual_cl:.8f} (target {target_cl:.8f}, dev = {(actual_cl/target_cl-1)*1e6:+.1f} ppm)')
    print()

# RMS evolution: how does each generation's signal change through levels?
print('Per-level RMS (a5=0 sectors):')
print(f'  {"Channel":>7s}  {"k":>2s}  {"RMS_g1":>12s}  {"RMS_g2":>12s}  {"g1/g1_R0":>10s}  {"g2/g2_R0":>10s}')
for ch_name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    g1_R0 = w0_rms[0, ch_a3, a7g1, 0]
    g2_R0 = w0_rms[0, ch_a3, a7g2, 0]
    for k in range(4):
        g1 = w0_rms[0, ch_a3, a7g1, k]
        g2 = w0_rms[0, ch_a3, a7g2, k]
        print(f'  {ch_name:>7s}  R{k}  {g1:12.6f}  {g2:12.6f}  {g1/g1_R0:10.4f}  {g2/g2_R0:10.4f}')
    print()

S4: PER-LEVEL CASCADE PROGRESSION
JAX device: CPU (1 device(s))
JAX warmup: 1.0s
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 1.65s
Integrated 210 branches in 1.7s

Per-level exponents:
  Channel   k      C0(Rk)       x(Rk)   x(Rk)/x(R0)
    QUARK  R0    189.1119    0.571450    1.00000000
    QUARK  R1     58.8635    0.735109    1.28639385
    QUARK  R2     39.8014    0.813195    1.42303924
    QUARK  R3      6.6067    1.586646    2.77652911
    QUARK  cross-level = 2.77652911 (target 2.77777778, dev = -449.5 ppm)

   LEPTON  R0      8.7738    2.454953    1.00000000
   LEPTON  R1      5.4299    3.151213    1.28361452
   LEPTON  R2      5.2273    3.223663    1.31312639
   LEPTON  R3      5.9120    3.000376    1.22217252
   LEPTON  cross-level = 1.22217252 (target 1.22222222, dev = -40.7 ppm)

Per-level RMS (a5=0 sectors):
  Channel   k        RMS_g1        RMS_g2    g1/g1_R0    g2/g2_R0
    QUARK  R0      2.075590      0.010975      1.0000      1.0000
    QUARK  R1   

## S5: Per-Level Cross-Level Decomposition

The total cross-level factor x(R₃)/x(R₀) can be decomposed into three steps: R₀→R₁, R₁→R₂, R₂→R₃, governed by primes p₁=2, p₂=3, p₃=5 respectively. We examine the per-step amplification to understand what each prime contributes.

In [12]:
# ── S5: Per-level cross-level decomposition ──

print('S5: PER-LEVEL CROSS-LEVEL DECOMPOSITION')
print('='*70)

# Per-level x values and step ratios
print(f'\n{"Channel":>7s}  {"Step":>8s}  {"x(Rk)":>10s}  {"x(Rk+1)":>10s}  {"step_CL":>10s}  {"prime":>6s}')
for ch_name in ['QUARK', 'LEPTON']:
    mass_r = M_S_D if ch_name == 'QUARK' else M_MU_E
    xs = []
    for k in range(4):
        c0 = w0_cp[ch_name][k]
        x = np.log(mass_r) / np.log(c0)
        xs.append(x)
    for k in range(3):
        step_cl = xs[k+1] / xs[k]
        pk = SA.primes[k]
        print(f'  {ch_name:>7s}  R{k}->R{k+1}  {xs[k]:10.6f}  {xs[k+1]:10.6f}  {step_cl:10.6f}  p{k+1}={pk}')
    total_cl = xs[3] / xs[0]
    product = np.prod([xs[k+1]/xs[k] for k in range(3)])
    print(f'  {ch_name:>7s}  Product of steps = {product:.8f} (= total CL = {total_cl:.8f})')
    print()

# RMS amplification per level
print('RMS amplification factor (RMS_k / RMS_{k-1}) per sector:')
print(f'  {"Channel":>7s}  {"Step":>8s}  {"amp_g1":>10s}  {"amp_g2":>10s}  {"ratio":>10s}')
for ch_name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    for k in range(3):
        g1_lo = w0_rms[0, ch_a3, a7g1, k]
        g1_hi = w0_rms[0, ch_a3, a7g1, k+1]
        g2_lo = w0_rms[0, ch_a3, a7g2, k]
        g2_hi = w0_rms[0, ch_a3, a7g2, k+1]
        amp1 = g1_hi / g1_lo
        amp2 = g2_hi / g2_lo
        print(f'  {ch_name:>7s}  R{k}->R{k+1}  {amp1:10.4f}  {amp2:10.4f}  {amp2/amp1:10.4f}')
    print()

# C0 progression
print('C0 progression:')
print(f'  {"Channel":>7s}  {"k":>2s}  {"C0(Rk)":>12s}  {"C0 ratio":>12s}')
for ch_name in ['QUARK', 'LEPTON']:
    c0s = [w0_cp[ch_name][k] for k in range(4)]
    for k in range(4):
        ratio_str = f'{c0s[k]/c0s[k-1]:.6f}' if k > 0 else '—'
        print(f'  {ch_name:>7s}  R{k}  {c0s[k]:12.4f}  {ratio_str:>12s}')
    print()

# Key: the cross-level = product of step factors
# Can we connect step factors to primes?
print('Step CL vs prime ratios:')
for ch_name in ['QUARK', 'LEPTON']:
    mass_r = M_S_D if ch_name == 'QUARK' else M_MU_E
    xs = [np.log(mass_r) / np.log(w0_cp[ch_name][k]) for k in range(4)]
    for k in range(3):
        step_cl = xs[k+1] / xs[k]
        pk = SA.primes[k]
        print(f'  {ch_name:>7s}  R{k}->R{k+1}: step_CL = {step_cl:.6f}, '
              f'pk/(pk-1) = {pk/(pk-1):.6f}, pk/(pk+1) = {pk/(pk+1):.6f}, '
              f'(pk+1)/pk = {(pk+1)/pk:.6f}')

S5: PER-LEVEL CROSS-LEVEL DECOMPOSITION

Channel      Step       x(Rk)     x(Rk+1)     step_CL   prime
    QUARK  R0->R1    0.571450    0.735109    1.286394  p1=2
    QUARK  R1->R2    0.735109    0.813195    1.106224  p2=3
    QUARK  R2->R3    0.813195    1.586646    1.951126  p3=5
    QUARK  Product of steps = 2.77652911 (= total CL = 2.77652911)

   LEPTON  R0->R1    2.454953    3.151213    1.283615  p1=2
   LEPTON  R1->R2    3.151213    3.223663    1.022991  p2=3
   LEPTON  R2->R3    3.223663    3.000376    0.930735  p3=5
   LEPTON  Product of steps = 1.22217252 (= total CL = 1.22217252)

RMS amplification factor (RMS_k / RMS_{k-1}) per sector:
  Channel      Step      amp_g1      amp_g2       ratio
    QUARK  R0->R1      0.7798      2.5054      3.2127
    QUARK  R1->R2      1.0732      1.5872      1.4789
    QUARK  R2->R3      1.0630      6.4037      6.0244

   LEPTON  R0->R1      2.6491      4.2805      1.6158
   LEPTON  R1->R2      1.5278      1.5870      1.0388
   LEPTON  R2->R3

## S6: Full Sector Landscape — All a₅=0 Pairs

The CP pairs use specific (a₃,a₇) sectors. To understand the cross-level mechanism, we examine ALL 12 a₅=0 sectors' RMS evolution through levels. This reveals whether the cross-level is pair-specific or reflects a broader pattern in how the cascade redistributes energy across CRT sectors.

In [14]:
# ── S6: Full a5=0 sector landscape ──

print('S6: FULL SECTOR LANDSCAPE (a5=0)')
print('='*70)

# All 12 a5=0 sectors: their crossing times and per-level RMS
print(f'\n{"(a3,a7)":>8s} {"ci":>5s} {"RMS_R0":>10s} {"RMS_R1":>10s} {"RMS_R2":>10s} {"RMS_R3":>10s}  {"ln(R0)":>8s} {"ln(R3)":>8s}  role')
for a3v in [0, 1]:
    for a7v in range(6):
        key = (a3v, 0, a7v)
        if key not in sector_ci:
            continue
        cv = sector_ci[key]
        rms = [w0_rms[0, a3v, a7v, k] for k in range(4)]
        role = ''
        if (a3v, a7v) == (1, 4): role = 'Q-g1'
        elif (a3v, a7v) == (1, 2): role = 'Q-g2'
        elif (a3v, a7v) == (0, 1): role = 'L-g1'
        elif (a3v, a7v) == (0, 5): role = 'L-g2'
        print(f'  ({a3v},{a7v})   {cv:5d} {rms[0]:10.6f} {rms[1]:10.6f} {rms[2]:10.6f} {rms[3]:10.6f}'
              f'  {np.log(rms[0]):8.4f} {np.log(rms[3]):8.4f}  {role}')

# ln(C0) for all possible pairs within each a3 class
print('\nAll sector pair exponents within a3 classes:')
for a3v in [0, 1]:
    a3_label = 'a3=0 (lepton class)' if a3v == 0 else 'a3=1 (quark class)'
    print(f'\n  {a3_label}:')
    # Collect sectors sorted by ci
    sectors = [(a7v, sector_ci[(a3v, 0, a7v)]) for a7v in range(6)
               if (a3v, 0, a7v) in sector_ci]
    sectors.sort(key=lambda x: x[1])
    
    # For each pair (i < j), compute ln(C0) and x at R0 and R3
    for i in range(len(sectors)):
        for j in range(i+1, len(sectors)):
            a7i, ci_i = sectors[i]
            a7j, ci_j = sectors[j]
            for mass_r, mass_label in [(M_S_D, 's/d'), (M_MU_E, 'mu/e')]:
                rms_i = [w0_rms[0, a3v, a7i, k] for k in range(4)]
                rms_j = [w0_rms[0, a3v, a7j, k] for k in range(4)]
                c0_R0 = rms_i[0] / rms_j[0]
                c0_R3 = rms_i[3] / rms_j[3]
                if c0_R0 > 1 and c0_R3 > 1:
                    x_R0 = np.log(mass_r) / np.log(c0_R0)
                    x_R3 = np.log(mass_r) / np.log(c0_R3)
                    cl = x_R3 / x_R0
                    # Only show if meaningful
                    if abs(cl) < 10:
                        flag = ''
                        if a3v == 1 and (a7i, a7j) == (4, 2) and mass_label == 's/d':
                            flag = ' ** QUARK CP **'
                        if a3v == 0 and (a7i, a7j) == (1, 5) and mass_label == 'mu/e':
                            flag = ' ** LEPTON CP **'
                        print(f'    ci={ci_i:3d}/{ci_j:3d} (a7={a7i},{a7j}) M={mass_label}: '
                              f'x(R0)={x_R0:7.4f}, x(R3)={x_R3:7.4f}, CL={cl:7.4f}{flag}')

# Focus: cross-level for ALL pairs in each a3 class (using their natural mass ratio)
print('\nCross-level for all consecutive pairs (using M_s/M_d for a3=1, M_mu/M_e for a3=0):')
for a3v in [0, 1]:
    mass_r = M_MU_E if a3v == 0 else M_S_D
    sectors = [(a7v, sector_ci[(a3v, 0, a7v)]) for a7v in range(6)
               if (a3v, 0, a7v) in sector_ci]
    sectors.sort(key=lambda x: x[1])
    for i in range(len(sectors)):
        for j in range(i+1, len(sectors)):
            a7i, ci_i = sectors[i]
            a7j, ci_j = sectors[j]
            rms_i = [w0_rms[0, a3v, a7i, k] for k in range(4)]
            rms_j = [w0_rms[0, a3v, a7j, k] for k in range(4)]
            c0_R0 = rms_i[0] / rms_j[0]
            c0_R3 = rms_i[3] / rms_j[3]
            if c0_R0 > 1 and c0_R3 > 1:
                cl = np.log(c0_R0) / np.log(c0_R3)
                print(f'    a3={a3v} ci={ci_i:3d}/{ci_j:3d} gap={ci_j-ci_i:3d}: CL = {cl:.6f}')

S6: FULL SECTOR LANDSCAPE (a5=0)

 (a3,a7)    ci     RMS_R0     RMS_R1     RMS_R2     RMS_R3    ln(R0)   ln(R3)  role
  (0,0)       1   0.296767   0.470840   0.921373   1.380226   -1.2148   0.3222  
  (0,1)      31   0.516338   1.367818   2.089689   1.973601   -0.6610   0.6799  L-g1
  (0,2)     121   0.010263   0.032215   0.025587   0.250832   -4.5792  -1.3830  
  (0,3)     181   0.010970   0.027544   0.036119   0.265691   -4.5126  -1.3254  
  (0,4)     151   0.010888   0.028128   0.034547   0.263506   -4.5201  -1.3337  
  (0,5)      61   0.058850   0.251905   0.399765   0.333832   -2.8328  -1.0971  L-g2
  (1,0)      71   0.026539   0.148617   0.198686   0.585814   -3.6292  -0.5348  
  (1,1)     101   0.008545   0.045123   0.019326   0.330830   -4.7625  -1.1061  
  (1,2)     191   0.010975   0.027498   0.043644   0.279486   -4.5121  -1.2748  Q-g2
  (1,3)      41   0.255177   0.772792   1.308738   2.076132   -1.3658   0.7305  
  (1,4)      11   2.075590   1.618601   1.737103   1.846494 

## S7: The Nonlinear Filter — Angular Forcing vs Linear Coupling

The cascade ODE for R_k has two driving terms from level k-1:
1. **Linear coupling**: κ·R_{k-1}/p_{k-1} — transfers amplitude proportionally
2. **Angular forcing difference**: ε·(sin(θ_k) - sin(θ_{k-1})/p_{k-1}) — depends nonlinearly on R_{k-1}

When R_{k-1} is large, θ_k = (R_{k-1} + θ_{k-1})/p_{k-1} varies over many radians → sin(θ_k) oscillates rapidly between branches → the RMS contribution averages down. When R_{k-1} is small, θ_k tracks θ_{k-1}/p_{k-1} smoothly → coherent forcing.

This creates a **saturating nonlinear filter**: large signals get damped relative to small signals, compressing C₀ at each level. We test this by separating linear vs angular contributions.

In [16]:
# ── S7: Per-branch R values at physical crossings ──

print('S7: NONLINEAR FILTER — BRANCH-RESOLVED ANALYSIS')
print('='*70)

# Look at per-branch R values at the 4 physical crossings
phys_cis = {
    'Q-g1': 11, 'Q-g2': 191, 'L-g1': 31, 'L-g2': 61
}

# Get indices of physical crossings in cis array
ci_idx = {name: np.where(cis == ci)[0][0] for name, ci in phys_cis.items()}

# For each physical crossing, collect R values across all 210 branches
print('\nBranch spread at physical crossings:')
print(f'  {"crossing":>6s}  {"ci":>4s}  {"level":>5s}  {"mean(R)":>10s}  {"std(R)":>10s}  {"RMS(R)":>10s}  {"R_wrapped":>10s}')
for name in ['Q-g1', 'Q-g2', 'L-g1', 'L-g2']:
    idx = ci_idx[name]
    for k in range(4):
        R_vals = np.array([res[b][idx, k] for b in all_branches])
        R_wrapped = np.mod(R_vals, 2*np.pi)
        R_wrapped[R_wrapped > np.pi] -= 2*np.pi
        print(f'  {name:>6s}  {phys_cis[name]:4d}  R{k:1d}     {R_vals.mean():10.4f}  {R_vals.std():10.4f}  '
              f'{np.sqrt(np.mean(R_wrapped**2)):10.6f}  [{R_wrapped.min():.3f}, {R_wrapped.max():.3f}]')
    print()

# Key test: how many radians does R0 span for each branch at each crossing?
print('R₀ at physical crossings (j1=0 vs j1=1):')
for name in ['Q-g1', 'Q-g2', 'L-g1', 'L-g2']:
    ci = phys_cis[name]
    r0_j0 = R0_at_crossing(float(ci), 0)
    r0_j1 = R0_at_crossing(float(ci), 1)
    # theta_1 = (R0 + theta_0) / p1. At integer ci, theta_0 = 2*pi*ci
    theta0 = OMEGA * ci  # = 2*pi*ci
    theta1_j0 = (r0_j0 + theta0) / p1
    theta1_j1 = (r0_j1 + theta0) / p1
    delta_theta1 = theta1_j1 - theta1_j0  # = (r0_j1 - r0_j0) / p1 = pi
    sin_spread = abs(np.sin(theta1_j1) - np.sin(theta1_j0))
    print(f'  {name}: R0(j1=0)={r0_j0:.6f}, R0(j1=1)={r0_j1:.6f}, '
          f'spread={r0_j1-r0_j0:.4f}')
    print(f'    theta1(j0)={theta1_j0:.4f} mod 2pi={theta1_j0%(2*np.pi):.4f}, '
          f'theta1(j1)={theta1_j1:.4f} mod 2pi={theta1_j1%(2*np.pi):.4f}')
    print(f'    sin(theta1) spread: {sin_spread:.6f}')

# Now look at how R1 depends on the branch structure
# The 210 branches decompose as (j1, j2, j3, j4) with j1 in {0,1}
# At R1 level, the relevant degrees of freedom are j1 and j2
print('\nR₁ per (j1,j2) group at each crossing (averaged over j3,j4):')
for name in ['Q-g1', 'Q-g2', 'L-g1', 'L-g2']:
    idx = ci_idx[name]
    print(f'  {name} (ci={phys_cis[name]}):')
    for j1 in range(p1):
        for j2 in range(p2):
            # Filter branches with this j1,j2
            branches_jj = [b for b in all_branches if b[0]==j1 and b[1]==j2]
            R1_vals = [res[b][idx, 1] for b in branches_jj]
            R1_wrapped = np.mod(np.array(R1_vals), 2*np.pi)
            R1_wrapped[R1_wrapped > np.pi] -= 2*np.pi
            R1_rms = np.sqrt(np.mean(R1_wrapped**2))
            print(f'    j1={j1},j2={j2}: R1_rms={R1_rms:.6f} ({len(branches_jj)} branches)')
    print()

S7: NONLINEAR FILTER — BRANCH-RESOLVED ANALYSIS

Branch spread at physical crossings:
  crossing    ci  level     mean(R)      std(R)      RMS(R)   R_wrapped
    Q-g1    11  R0         1.4647      1.4706    2.075590  [-0.006, 2.935]
    Q-g1    11  R1         3.5156      2.4608    1.618601  [-2.230, 2.978]
    Q-g1    11  R2         6.6514      4.2059    1.737103  [-3.098, 3.029]
    Q-g1    11  R3         9.6948      5.8909    1.846494  [-3.132, 3.118]

    Q-g2   191  R0        -0.0110      0.0000    0.010975  [-0.011, -0.011]
    Q-g2   191  R1         0.0275      0.0000    0.027498  [0.027, 0.028]
    Q-g2   191  R2        -0.0436      0.0001    0.043644  [-0.044, -0.043]
    Q-g2   191  R3         0.2795      0.0001    0.279486  [0.279, 0.280]

    L-g1    31  R0         0.3602      0.3699    0.516338  [-0.010, 0.730]
    L-g1    31  R1         1.1628      0.7202    1.367818  [0.031, 2.295]
    L-g1    31  R2         2.1401      1.1461    2.089689  [-3.051, 3.024]
    L-g1    31  

## S8: Coherence Saturation — The Circle Limit

S7 revealed the mechanism: sectors with large R₀ (transient regime) spread across the circle at higher levels, reaching a saturated RMS ≈ π/√3 ≈ 1.814. Sectors with small R₀ (SS regime) stay coherent and grow through linear coupling.

The cross-level is entirely generated by the **nonlinear angular forcing** (sin(θ_k) terms). Pure linear coupling (κ·R_{k-1}/p_k) would preserve C₀ exactly. The sin terms create amplitude-dependent phase spreading that saturates large signals while coherently amplifying small ones.

**Key prediction**: If the transient sector RMS saturates at π/√3, then C₀(R₃) ≈ (π/√3)/RMS_g2(R₃), and the cross-level depends only on how much the SS sector grows through the cascade.

In [18]:
# ── S8: Coherence saturation and the circle limit ──

print('S8: COHERENCE SATURATION — THE CIRCLE LIMIT')
print('='*70)

UNIFORM_RMS = np.pi / np.sqrt(3)
print(f'Uniform circle limit: pi/sqrt(3) = {UNIFORM_RMS:.6f}')

# Check RMS vs uniform limit for all a5=0 sectors at each level
print(f'\nRMS / (pi/sqrt(3)) at each level:')
print(f'  {"(a3,a7)":>8s} {"ci":>5s} {"R0":>8s} {"R1":>8s} {"R2":>8s} {"R3":>8s}  role')
for a3v in [0, 1]:
    for a7v in range(6):
        key = (a3v, 0, a7v)
        if key not in sector_ci:
            continue
        cv = sector_ci[key]
        ratios = [w0_rms[0, a3v, a7v, k] / UNIFORM_RMS for k in range(4)]
        role = ''
        if (a3v, a7v) == (1, 4): role = 'Q-g1'
        elif (a3v, a7v) == (1, 2): role = 'Q-g2'
        elif (a3v, a7v) == (0, 1): role = 'L-g1'
        elif (a3v, a7v) == (0, 5): role = 'L-g2'
        print(f'  ({a3v},{a7v})   {cv:5d} {ratios[0]:8.4f} {ratios[1]:8.4f} {ratios[2]:8.4f} {ratios[3]:8.4f}  {role}')

# Branch distribution analysis at each level
print('\nBranch distribution at R₃ for each physical crossing:')
for name in ['Q-g1', 'Q-g2', 'L-g1', 'L-g2']:
    idx = ci_idx[name]
    R_vals = np.array([res[b][idx, 3] for b in all_branches])
    R_wrapped = np.mod(R_vals, 2*np.pi)
    R_wrapped[R_wrapped > np.pi] -= 2*np.pi
    rms = np.sqrt(np.mean(R_wrapped**2))
    
    # Binned distribution
    n_bins = 12
    hist, edges = np.histogram(R_wrapped, bins=n_bins, range=(-np.pi, np.pi))
    uniformity = np.std(hist) / np.mean(hist)  # CV: 0 = perfectly uniform
    
    print(f'  {name}: RMS={rms:.4f}, RMS/uniform={rms/UNIFORM_RMS:.4f}, '
          f'CV(hist)={uniformity:.3f} (0=uniform)')

# KEY: What determines the SS sector's R₃ amplitude?
# For the coherent (SS) sector, the cascade is approximately linear:
# R_{k,SS} ~ sum of linear contributions from each level
print('\n\nCOHERENCE-BASED C₀ PREDICTION:')
print('If transient sectors saturate at pi/sqrt(3), then:')
for ch_name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    rms_g1_R3 = w0_rms[0, ch_a3, a7g1, 3]
    rms_g2_R3 = w0_rms[0, ch_a3, a7g2, 3]
    c0_actual = rms_g1_R3 / rms_g2_R3
    c0_uniform = UNIFORM_RMS / rms_g2_R3
    
    mass_r = M_S_D if ch_name == 'QUARK' else M_MU_E
    x_actual = np.log(mass_r) / np.log(c0_actual)
    x_uniform = np.log(mass_r) / np.log(c0_uniform)
    
    print(f'\n  {ch_name}:')
    print(f'    RMS_g1(R3) = {rms_g1_R3:.6f}, RMS_g1/uniform = {rms_g1_R3/UNIFORM_RMS:.4f}')
    print(f'    RMS_g2(R3) = {rms_g2_R3:.6f}')
    print(f'    C0 actual  = {c0_actual:.4f} -> x = {x_actual:.6f}')
    print(f'    C0 uniform = {c0_uniform:.4f} -> x = {x_uniform:.6f}')
    print(f'    If g1 were exactly uniform: x changes by {(x_uniform/x_actual-1)*1e6:+.0f} ppm')

# The linear cascade prediction for coherent sector
print('\n\nLINEAR CASCADE FOR COHERENT SECTOR:')
print('In pure linear coupling, dR_k/dt + kappa*R_k = kappa*R_{k-1}/p_{k-1}')
print('Steady-state: R_k(SS) = R_{k-1}(SS)/p_{k-1}')
print('So RMS_g2 would be: R0 -> R0/2 -> R0/6 -> R0/30')
for ch_name, (ch_a3, a7g1, a7g2) in CP_PAIRS.items():
    rms_g2_R0 = w0_rms[0, ch_a3, a7g2, 0]
    linear_pred = [rms_g2_R0]
    for pk in SA.primes[:3]:
        linear_pred.append(linear_pred[-1] / pk)
    actual = [w0_rms[0, ch_a3, a7g2, k] for k in range(4)]
    print(f'  {ch_name} g2 sector:')
    print(f'    Linear prediction: {" -> ".join(f"{v:.6f}" for v in linear_pred)}')
    print(f'    Actual RMS:        {" -> ".join(f"{v:.6f}" for v in actual)}')
    print(f'    Actual/Linear:     {" -> ".join(f"{a/l:.4f}" for a,l in zip(actual, linear_pred))}')
    # The ratio actual/linear shows the nonlinear boost
    boost_R3 = actual[3] / linear_pred[3]
    print(f'    Total nonlinear boost at R3: {boost_R3:.2f}x')

S8: COHERENCE SATURATION — THE CIRCLE LIMIT
Uniform circle limit: pi/sqrt(3) = 1.813799

RMS / (pi/sqrt(3)) at each level:
   (a3,a7)    ci       R0       R1       R2       R3  role
  (0,0)       1   0.1636   0.2596   0.5080   0.7610  
  (0,1)      31   0.2847   0.7541   1.1521   1.0881  L-g1
  (0,2)     121   0.0057   0.0178   0.0141   0.1383  
  (0,3)     181   0.0060   0.0152   0.0199   0.1465  
  (0,4)     151   0.0060   0.0155   0.0190   0.1453  
  (0,5)      61   0.0324   0.1389   0.2204   0.1841  L-g2
  (1,0)      71   0.0146   0.0819   0.1095   0.3230  
  (1,1)     101   0.0047   0.0249   0.0107   0.1824  
  (1,2)     191   0.0061   0.0152   0.0241   0.1541  Q-g2
  (1,3)      41   0.1407   0.4261   0.7215   1.1446  
  (1,4)      11   1.1443   0.8924   0.9577   1.0180  Q-g1
  (1,5)     131   0.0059   0.0165   0.0208   0.1588  

Branch distribution at R₃ for each physical crossing:
  Q-g1: RMS=1.8465, RMS/uniform=1.0180, CV(hist)=0.826 (0=uniform)
  Q-g2: RMS=0.2795, RMS/uniform=

## S9: Rational Approximation and Self-Referential Arithmetic

The computed quantities x(R₀), x(R₃), and cross-level are transcendental — they involve exp(-ci/√210) and ln(mass ratio). Yet they approximate rationals built from {2,3,5,7} with surprising precision:
- x_q(R₀) ≈ 4/7 (37 ppm), cross-level ≈ 25/9 (450 ppm), x_q(R₃) ≈ 100/63 (413 ppm)
- x_l(R₀) ≈ 27/11 (166 ppm), cross-level ≈ 11/9 (41 ppm), x_l(R₃) ≈ 3 (125 ppm)

Note: 100/63 = (2²·5²)/(3²·7) and 3 = p₂ — both use only the solenoid primes. We test whether the rational closeness is special by examining continued fraction convergents and checking if {2,3,5,7}-smooth rationals appear preferentially.

In [20]:
# ── S9: Rational approximation and self-referential arithmetic ──

print('S9: RATIONAL APPROXIMATION — CONTINUED FRACTIONS')
print('='*70)

def continued_fraction(x, max_terms=12):
    """Return CF coefficients [a0; a1, a2, ...]."""
    cf = []
    for _ in range(max_terms):
        a = int(np.floor(x))
        cf.append(a)
        frac = x - a
        if frac < 1e-12:
            break
        x = 1.0 / frac
    return cf

def cf_convergents(cf):
    """Return list of (p, q) convergents from CF coefficients."""
    convs = []
    h_prev, h_curr = 0, 1
    k_prev, k_curr = 1, 0
    for a in cf:
        h_prev, h_curr = h_curr, a * h_curr + h_prev
        k_prev, k_curr = k_curr, a * k_curr + k_prev
        convs.append((h_curr, k_curr))
    return convs

def is_smooth(n, primes=(2, 3, 5, 7)):
    """Check if n is smooth w.r.t. given primes."""
    n = abs(n)
    if n == 0:
        return False
    for p in primes:
        while n % p == 0:
            n //= p
    return n == 1

# Key quantities
quantities = {}
for ch_name in ['QUARK', 'LEPTON']:
    mass_r = M_S_D if ch_name == 'QUARK' else M_MU_E
    xs = [np.log(mass_r) / np.log(w0_cp[ch_name][k]) for k in range(4)]
    cl = xs[3] / xs[0]
    quantities[f'x_{ch_name[:1].lower()}(R0)'] = xs[0]
    quantities[f'x_{ch_name[:1].lower()}(R3)'] = xs[3]
    quantities[f'CL_{ch_name[:1].lower()}'] = cl

print('\nContinued fraction analysis:')
for name, val in quantities.items():
    cf = continued_fraction(val)
    convs = cf_convergents(cf)
    print(f'\n  {name} = {val:.10f}')
    print(f'    CF = [{cf[0]}; {", ".join(str(c) for c in cf[1:])}]')
    print(f'    Convergents:')
    for p, q in convs[:7]:
        dev = (val / (p/q) - 1) * 1e6
        smooth_p = is_smooth(p)
        smooth_q = is_smooth(q)
        smooth_tag = ''
        if smooth_p and smooth_q:
            smooth_tag = '  ** {2,3,5,7}-smooth **'
        elif smooth_p or smooth_q:
            smooth_tag = f'  ({"num" if smooth_p else "den"} smooth)'
        print(f'      {p}/{q} = {p/q:.10f}  ({dev:+.1f} ppm){smooth_tag}')

# Direct check: are the target rationals the BEST {2,3,5,7}-smooth approximants?
print('\n\nBEST {2,3,5,7}-SMOOTH RATIONAL APPROXIMANTS:')
for name, val in quantities.items():
    best_dev = float('inf')
    best_frac = None
    # Search all p/q with small p,q where both are 7-smooth
    for q in range(1, 200):
        if not is_smooth(q):
            continue
        p = round(val * q)
        if p > 0 and is_smooth(p):
            dev = abs(val / (p/q) - 1)
            if dev < best_dev:
                best_dev = dev
                best_frac = (p, q)
    p, q = best_frac
    from math import gcd
    g = gcd(p, q)
    p, q = p // g, q // g
    print(f'  {name} = {val:.8f} ~ {p}/{q} = {p/q:.8f} ({best_dev*1e6:.1f} ppm)')

# The factored product check
print('\n\nFACTORED PRODUCT CHECK:')
for ch_name in ['QUARK', 'LEPTON']:
    mass_r = M_S_D if ch_name == 'QUARK' else M_MU_E
    xs = [np.log(mass_r) / np.log(w0_cp[ch_name][k]) for k in range(4)]
    x_R3 = xs[3]
    x_R0 = xs[0]
    cl = xs[3] / xs[0]
    
    if ch_name == 'QUARK':
        target_x0, target_cl, target_x3 = Fraction(4,7), Fraction(25,9), Fraction(100,63)
    else:
        target_x0, target_cl, target_x3 = Fraction(27,11), Fraction(11,9), Fraction(3,1)
    
    dev_x0 = (x_R0 / float(target_x0) - 1) * 1e6
    dev_cl = (cl / float(target_cl) - 1) * 1e6
    dev_x3 = (x_R3 / float(target_x3) - 1) * 1e6
    
    print(f'\n  {ch_name}:')
    print(f'    x(R0) = {x_R0:.8f} ~ {target_x0} = {float(target_x0):.8f} ({dev_x0:+.1f} ppm)')
    print(f'    CL    = {cl:.8f} ~ {target_cl} = {float(target_cl):.8f} ({dev_cl:+.1f} ppm)')
    print(f'    x(R3) = {x_R3:.8f} ~ {target_x3} = {float(target_x3):.8f} ({dev_x3:+.1f} ppm)')
    print(f'    Product: ({target_x0})({target_cl}) = {target_x0 * target_cl} = {float(target_x0 * target_cl):.8f}')
    
    # Factor the target into primes
    def prime_factor(n):
        factors = {}
        for p in [2, 3, 5, 7, 11, 13]:
            while n % p == 0:
                factors[p] = factors.get(p, 0) + 1
                n //= p
        if n > 1:
            factors[n] = 1
        return factors
    
    print(f'    {target_x3} = ', end='')
    pf = prime_factor(target_x3.numerator)
    numer_str = ' * '.join(f'{p}^{e}' if e > 1 else str(p) for p, e in sorted(pf.items()))
    pf = prime_factor(target_x3.denominator)
    if pf:
        denom_str = ' * '.join(f'{p}^{e}' if e > 1 else str(p) for p, e in sorted(pf.items()))
        print(f'({numer_str}) / ({denom_str})')
    else:
        print(numer_str)

S9: RATIONAL APPROXIMATION — CONTINUED FRACTIONS

Continued fraction analysis:

  x_q(R0) = 0.5714495813
    CF = [0; 1, 1, 2, 1, 970, 1, 1, 1, 4, 1, 19]
    Convergents:
      0/1 = 0.0000000000  (+inf ppm)  (den smooth)
      1/1 = 1.0000000000  (-428550.4 ppm)  ** {2,3,5,7}-smooth **
      1/2 = 0.5000000000  (+142899.2 ppm)  ** {2,3,5,7}-smooth **
      3/5 = 0.6000000000  (-47584.0 ppm)  ** {2,3,5,7}-smooth **
      4/7 = 0.5714285714  (+36.8 ppm)  ** {2,3,5,7}-smooth **
      3883/6795 = 0.5714495953  (-0.0 ppm)
      3887/6802 = 0.5714495737  (+0.0 ppm)

  x_q(R3) = 1.5866463961
    CF = [1; 1, 1, 2, 2, 1, 1, 2, 8, 2, 1, 1]
    Convergents:
      1/1 = 1.0000000000  (+586646.4 ppm)  ** {2,3,5,7}-smooth **
      2/1 = 2.0000000000  (-206676.8 ppm)  ** {2,3,5,7}-smooth **
      3/2 = 1.5000000000  (+57764.3 ppm)  ** {2,3,5,7}-smooth **
      8/5 = 1.6000000000  (-8346.0 ppm)  ** {2,3,5,7}-smooth **
      19/12 = 1.5833333333  (+2092.5 ppm)  (den smooth)
      27/17 = 1.5882352941 

## S10: Synthesis — Answers to GAP-02 Sub-Questions

### Q1: WHY x_q(R₀) = 4/7?

**Answer**: x(R₀) = ln(M)/ln(C₀(R₀)) where C₀ is set by the analytic R₀ formula at the CP pair crossings. For the quark (ci=11 vs ci=191), this is the ratio of a transient-regime amplitude to a steady-state offset. The transcendental value 0.57145 has continued fraction [0; 1, 1, 2, 1, **970**, ...] — the enormous partial quotient 970 means 4/7 is an **exceptionally good rational approximation** (37 ppm). It is also the simplest {2,3,5,7}-smooth convergent. The value is set by exp(-11/√210) via the R₀ analytic formula.

### Q2: WHY cross-level = 25/9 for quarks, 11/9 for leptons?

**Answer**: The cross-level = ln(C₀(R₀))/ln(C₀(R₃)) measures how the cascade compresses the log-ratio. The mechanism is a **nonlinear saturating filter**:
- Large R₀ sectors (transient regime) → branches spread incoherently across the circle → RMS saturates near π/√3
- Small R₀ sectors (SS regime) → branches stay coherent → RMS grows by 170-764× above linear prediction

The specific values 25/9 and 11/9 are **CF convergents** of the transcendental cross-level values. Both are determined by the full nonlinear ODE dynamics with parameters κ = ε = 1/√210, ω = 2π, acting on the specific crossing times selected by CRT arithmetic. The cross-level is **pair-specific** (varies from 1.1 to 7.4 across different sector pairs) — it is NOT a universal cascade property.

25/9 (quark) is a smooth convergent at 450 ppm; 11/9 (lepton) is a convergent at 41 ppm.

### Q3: WHERE does 11 enter the lepton sector?

**Answer**: 11 enters through **continued fraction arithmetic**, not through algebraic structure. The transcendental x_l(R₀) = 2.4550 has CF convergent 27/11 (166 ppm), and the transcendental CL_l = 1.2222 has CF convergent 11/9 (41 ppm). Neither is {2,3,5,7}-smooth. But the product (27/11)(11/9) = 3 cancels the 11, producing the smooth value p₂ = 3.

The 11 is an **artifact of rational decomposition** — it appears when factoring x_l(R₃) ≈ 3 into x_l(R₀) × CL, but is not a fundamental property of the system. What IS fundamental is x_l(R₃) ≈ 3 = p₂, which has CF [3; **2660**, ...] — an astronomically good integer approximation.

### Q4: CAN 100/63 be derived from cascade equations?

**Answer**: Not from a closed-form derivation. 100/63 = (2²·5²)/(3²·7) is the best {2,3,5,7}-smooth rational approximant to x_q(R₃) = 1.5866 (413 ppm). It is NOT a CF convergent — the CF convergents are 1, 2, 3/2, 8/5, 19/12, 27/17, 46/29... None of these are smooth. The value 100/63 arises as the **product of two CF convergents** that ARE smooth: (4/7)(25/9).

The quark exponent is less precisely determined than the lepton: 413 ppm vs 125 ppm. This reflects the quark channel's sensitivity to the full nonlinear dynamics at the R₂→R₃ step (where the g2 sector gets amplified 6.4× and the step CL is 1.95).

### Summary of mechanism:

```
R₀ (analytic)              Cascade (nonlinear ODE)           x(R₃) (emergent)
─────────────              ────────────────────────           ──────────────────
Exponential transient      Saturating phase filter:           Transcendental values
at coprime crossings  →    large R → incoherent → capped  →  that approximate
sets C₀(R₀)               small R → coherent → amplified     {2,3,5,7}-rationals

QUARK: C₀=189              3 cascade steps (p=2,3,5)         x ≈ 100/63 (413 ppm)
  ci=11 transient           step CLs: 1.29, 1.11, 1.95       = (4/7)(25/9)
  ci=191 steady-state       total CL ≈ 25/9

LEPTON: C₀=8.8             3 cascade steps (p=2,3,5)         x ≈ 3 = p₂ (125 ppm)
  ci=31 transient           step CLs: 1.28, 1.02, 0.93       = (27/11)(11/9)
  ci=61 weak transient      total CL ≈ 11/9
```

In [22]:
# ── S10: Scorecard ──

print('S10: NB138 SCORECARD')
print('='*70)

print('''
GAP-02 Sub-questions                              Status
──────────────────                              ──────
Q1: WHY x_q(R0) = 4/7?                          ANSWERED
   → Analytic R0 formula gives transcendental value with
     CF convergent 4/7 at 37 ppm (pq 970 → exceptional fit).
     Set by exp(-11/sqrt(210)).

Q2: WHY cross-level = 25/9 (quark), 11/9 (lepton)?  MECHANISM IDENTIFIED
   → Nonlinear saturating filter: large R → incoherent,
     small R → coherent. Differential amplification produces
     pair-specific cross-levels. 25/9 and 11/9 are CF
     convergents of transcendental values (450/41 ppm).
     Not derivable in closed form from cascade equations.

Q3: WHERE does 11 enter the lepton sector?       ANSWERED
   → CF arithmetic artifact. x_l(R0) ~ 27/11, CL ~ 11/9,
     product (27/11)(11/9) = 3 cancels the 11.
     Fundamental quantity is x_l(R3) ~ 3 = p2 (125 ppm).

Q4: CAN 100/63 be derived from cascade equations?  ANSWERED (NEGATIVE)
   → No closed-form derivation. 100/63 is the best {2,3,5,7}-
     smooth approximant (413 ppm) arising as product of two
     CF convergents: (4/7)(25/9). Not itself a CF convergent.

NEW FINDINGS:
  • R0 is exactly solvable: oscillatory SS vanishes at integer crossings
  • CRT gives exactly 1 crossing per sector → CP ratio = 2-point comparison
  • Cascade = nonlinear phase filter: coherent sectors amplified 170-764x
    above linear prediction; incoherent sectors saturate near pi/sqrt(3)
  • Cross-level is pair-specific (1.1 to 7.4), not a universal cascade property
  • Large CF partial quotients (970, 2660) indicate self-referential arithmetic:
    system outputs approximate {2,3,5,7}-rationals from {2,3,5,7}-parameters
''')

print('Cells: S0-S10 (title + 10 sections, 21 cells)')
print('Identities: none new (mechanism paper, not identity discovery)')
print('Status: GAP-02 sub-questions RESOLVED')

S10: NB138 SCORECARD

GAP-02 Sub-questions                              Status
──────────────────                              ──────
Q1: WHY x_q(R0) = 4/7?                          ANSWERED
   → Analytic R0 formula gives transcendental value with
     CF convergent 4/7 at 37 ppm (pq 970 → exceptional fit).
     Set by exp(-11/sqrt(210)).

Q2: WHY cross-level = 25/9 (quark), 11/9 (lepton)?  MECHANISM IDENTIFIED
   → Nonlinear saturating filter: large R → incoherent,
     small R → coherent. Differential amplification produces
     pair-specific cross-levels. 25/9 and 11/9 are CF
     convergents of transcendental values (450/41 ppm).
     Not derivable in closed form from cascade equations.

Q3: WHERE does 11 enter the lepton sector?       ANSWERED
   → CF arithmetic artifact. x_l(R0) ~ 27/11, CL ~ 11/9,
     product (27/11)(11/9) = 3 cancels the 11.
     Fundamental quantity is x_l(R3) ~ 3 = p2 (125 ppm).

Q4: CAN 100/63 be derived from cascade equations?  ANSWERED (NEGATIVE)
   → No 